# LLM Benchmark on Google Colab T4 GPU

This notebook demonstrates how to:
1. Load a pre-trained LLM model
2. Run inference benchmarks on Google Colab's T4 GPU
3. Measure performance metrics (latency, throughput, memory usage)
4. Visualize the results

The model used is **DistilBERT** - a lightweight version of BERT that's perfect for testing on limited GPU resources like T4.

## 1. Install Required Libraries

In [1]:
# Install required libraries
import subprocess
import sys

# Install transformers and torch
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "datasets"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "matplotlib"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pandas"])

print("✓ All libraries installed successfully!")


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip


✓ All libraries installed successfully!



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip


## 2. Check GPU and Setup

In [2]:
import torch
import numpy as np
import time
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import matplotlib.pyplot as plt
import pandas as pd
from datetime import datetime

# Check GPU availability
print("GPU Information:")
print(f"  Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"  CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"  Current GPU Memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print()

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ Using device: {device}")

GPU Information:
  Device: CPU
  CUDA Available: False

✓ Using device: cpu


## 3. Load Pretrained LLM Model

We use **DistilBERT** - a lightweight BERT model that works great on T4 GPUs. It's 40% smaller and 60% faster than BERT while retaining 97% of its performance.

In [ ]:
# Load DistilBERT tokenizer and model
print("Loading DistilBERT model...")
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# Move model to GPU
model = model.to(device)
model.eval()  # Set to evaluation mode

print(f"✓ Model loaded successfully!")
print(f"  Model: {model_name}")
print(f"  Total parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")

# Get model size
param_size = sum(p.numel() for p in model.parameters()) * 4 / (1024 ** 2)  # 4 bytes per float32
print(f"  Model size: {param_size:.2f} MB")

Loading DistilBERT model...


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✓ Model loaded successfully!
  Model: distilbert-base-uncased
  Total parameters: 66.96M
  Model size: 255.41 MB


## 4. Load ARC (AI2 Reasoning Challenge) Dataset

The ARC dataset contains science exam questions with 4 multiple-choice answers. It tests reasoning and knowledge across science, physics, chemistry, and biology.

In [ ]:
from datasets import load_dataset

# Load ARC dataset (Easy and Challenge)
print("Loading ARC dataset...")
arc_easy = load_dataset("ai2_arc", "ARC-Easy", split="test", trust_remote_code=True)
arc_challenge = load_dataset("ai2_arc", "ARC-Challenge", split="test", trust_remote_code=True)

# Combine and limit to first 30 samples from each for T4 GPU
arc_data = []

for example in arc_easy[:15]:
    arc_data.append(example)

for example in arc_challenge[:15]:
    arc_data.append(example)

print(f"✓ Loaded {len(arc_data)} ARC questions (15 Easy + 15 Challenge)")
print(f"\nFirst example:")
print(f"  Question: {arc_data[0]['question']}")
print(f"  Choices: {arc_data[0]['choices']['text']}")
print(f"  Answer: {arc_data[0]['answerKey']}")

# Prepare data in format suitable for classification
arc_samples = []
for example in arc_data:
    correct_answer_idx = ord(example['answerKey']) - ord('A')  # Convert A,B,C,D to 0,1,2,3
    choices = example['choices']['text']
    
    # Create premise (question + each choice combination)
    for idx, choice in enumerate(choices):
        arc_samples.append({
            'question': example['question'],
            'choice': choice,
            'label': 1 if idx == correct_answer_idx else 0,  # 1 if correct, 0 if not
            'choice_idx': idx,
            'question_id': example['id']
        })

print(f"\n✓ Prepared {len(arc_samples)} samples for evaluation")
print(f"  Correct answers: {sum(s['label'] for s in arc_samples)}")

Created benchmark dataset with 60 samples
Sample texts (first 3):
  1. This movie is absolutely wonderful and I really enjoyed watc...
  2. The weather today is quite pleasant, perfect for outdoor act...
  3. I'm not satisfied with the quality of this product at all....

Tokenizing inputs...
✓ Tokenization complete!
  Input shape: torch.Size([60, 128])
  Max sequence length: 128 tokens


## 5. Run ARC Benchmark

Evaluate the model on ARC questions - measure accuracy and reasoning capability.

In [ ]:
# Evaluate model on ARC dataset
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
from collections import defaultdict

print("Running ARC Benchmark...\n")
print("=" * 60)

# Store predictions per question
question_predictions = defaultdict(list)
question_labels = defaultdict(list)
all_predictions = []
all_labels = []
timings = []

# Tokenize all samples at once for efficiency
question_choice_pairs = [f"{s['question']} {s['choice']}" for s in arc_samples]

print(f"Processing {len(arc_samples)} question-choice pairs...")
print(f"Batch processing with batch size: 8\n")

# Process in batches
batch_size = 8
correct_count = 0
total_count = 0

for batch_idx in range(0, len(arc_samples), batch_size):
    batch_samples = arc_samples[batch_idx:batch_idx + batch_size]
    batch_texts = [f"{s['question']} {s['choice']}" for s in batch_samples]
    
    # Tokenize
    encodings = tokenizer(
        batch_texts,
        truncation=True,
        padding="max_length",
        max_length=256,
        return_tensors="pt"
    )
    
    input_ids = encodings["input_ids"].to(device)
    attention_mask = encodings["attention_mask"].to(device)
    
    # Measure inference time
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    
    start_time = time.time()
    
    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits  # Shape: (batch_size, 2)
    
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    
    end_time = time.time()
    timings.append(end_time - start_time)
    
    # Get confidence scores for "correct" label (class 1)
    confidences = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
    
    # Store predictions grouped by question
    for i, sample in enumerate(batch_samples):
        q_id = sample['question_id']
        pred_confidence = confidences[i]
        
        question_predictions[q_id].append({
            'choice_idx': sample['choice_idx'],
            'confidence': pred_confidence,
            'label': sample['label']
        })

print(f"\n✓ Inference complete!")
print(f"Total inference time: {sum(timings):.2f}s")
print(f"Average time per sample: {(sum(timings) / len(arc_samples) * 1000):.2f}ms")

# Determine best answer per question and calculate accuracy
predictions_per_question = []
labels_per_question = []
per_sample_preds = []
per_sample_labels = []

for q_id, choices in question_predictions.items():
    # Find choice with highest confidence
    best_choice = max(choices, key=lambda x: x['confidence'])
    predicted_correct = best_choice['label']
    
    predictions_per_question.append(predicted_correct)
    
    # Find ground truth
    ground_truth = any(c['label'] == 1 for c in choices)
    labels_per_question.append(1 if ground_truth else 0)
    
    # Per-sample collection
    for choice in choices:
        per_sample_preds.append(choice['confidence'] > 0.5)
        per_sample_labels.append(choice['label'])

# Calculate metrics
accuracy_per_question = accuracy_score(labels_per_question, predictions_per_question)
accuracy_per_choice = accuracy_score(per_sample_labels, per_sample_preds)

print(f"\nAccuracy Metrics:")
print(f"  Per-Question Accuracy: {accuracy_per_question * 100:.2f}%")
print(f"  Per-Choice Accuracy: {accuracy_per_choice * 100:.2f}%")

benchmark_results = {
    'Accuracy (per question)': accuracy_per_question,
    'Accuracy (per choice)': accuracy_per_choice,
    'Total Questions': len(predictions_per_question),
    'Total Samples': len(arc_samples),
    'Avg Inference Time (ms)': (sum(timings) / len(arc_samples)) * 1000,
    'Total Inference Time (s)': sum(timings)
}

print("\n✓ Benchmark complete!")

Running inference benchmarks...

Testing batch size: 1
  Avg Latency: 81.70 ms ± 0.94 ms
  Throughput: 12.24 samples/s

Testing batch size: 4


KeyboardInterrupt: 

## 6. Display Results

Show detailed benchmark results including accuracy scores and performance metrics.

In [ ]:
# Display detailed results
print("\n" + "=" * 60)
print("ARC BENCHMARK RESULTS")
print("=" * 60)

print(f"\nDataset Statistics:")
print(f"  Total Questions: {benchmark_results['Total Questions']}")
print(f"  Total Samples: {benchmark_results['Total Samples']}")
print(f"  Question-Answer Pairs: {len(arc_data)}")

print(f"\nAccuracy Scores:")
print(f"  Per-Question Accuracy: {benchmark_results['Accuracy (per question)'] * 100:.2f}%")
print(f"  Per-Choice Accuracy: {benchmark_results['Accuracy (per choice)'] * 100:.2f}%")

print(f"\nPerformance Metrics:")
print(f"  Avg Inference Time: {benchmark_results['Avg Inference Time (ms)']:.2f} ms")
print(f"  Total Inference Time: {benchmark_results['Total Inference Time (s)']:.2f} s")
print(f"  Samples per Second: {benchmark_results['Total Samples'] / benchmark_results['Total Inference Time (s)']:.2f}")

if torch.cuda.is_available():
    peak_memory = torch.cuda.max_memory_allocated() / 1024 ** 2
    current_memory = torch.cuda.memory_allocated() / 1024 ** 2
    total_memory = torch.cuda.get_device_properties(0).total_memory / 1024 ** 2
    
    print(f"\nGPU Memory Usage:")
    print(f"  Peak Memory: {peak_memory:.2f} MB")
    print(f"  Current Memory: {current_memory:.2f} MB")
    print(f"  Total GPU Memory: {total_memory:.2f} MB")
    print(f"  Memory Utilization: {(peak_memory / total_memory) * 100:.2f}%")

print(f"\nModel Information:")
print(f"  Model: {model_name}")
print(f"  Total Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")
print(f"  Model Size: {param_size:.2f} MB")

print("\n" + "=" * 60)

## 7. Visualize Results

Create plots to display benchmark results for easy interpretation.

In [ ]:
# Create visualizations for ARC benchmark results
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("DistilBERT Performance on ARC (AI2 Reasoning Challenge)", fontsize=16, fontweight="bold")

# Plot 1: Accuracy comparison
ax1 = axes[0, 0]
accuracies = [benchmark_results['Accuracy (per question)'] * 100, 
              benchmark_results['Accuracy (per choice)'] * 100]
colors = ['steelblue', 'coral']
bars1 = ax1.bar(['Per-Question\nAccuracy', 'Per-Choice\nAccuracy'], accuracies, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
ax1.set_ylabel('Accuracy (%)', fontsize=11, fontweight="bold")
ax1.set_title('ARC Benchmark Accuracy', fontsize=12, fontweight="bold")
ax1.set_ylim([0, 100])
ax1.grid(True, alpha=0.3, axis='y')
for bar, acc in zip(bars1, accuracies):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height, f'{acc:.1f}%', 
            ha='center', va='bottom', fontsize=10, fontweight='bold')

# Plot 2: Inference performance
ax2 = axes[0, 1]
metrics = ['Avg Time\n(ms)', 'Samples/sec']
values = [benchmark_results['Avg Inference Time (ms)'], 
          benchmark_results['Total Samples'] / benchmark_results['Total Inference Time (s)']]
bars2 = ax2.bar(metrics, values, color=['darkgreen', 'darkorange'], alpha=0.7, edgecolor='black', linewidth=1.5)
ax2.set_ylabel('Value', fontsize=11, fontweight="bold")
ax2.set_title('Inference Performance', fontsize=12, fontweight="bold")
ax2.grid(True, alpha=0.3, axis='y')
for bar, val, metric in zip(bars2, values, metrics):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height, f'{val:.2f}', 
            ha='center', va='bottom', fontsize=10, fontweight='bold')

# Plot 3: Dataset breakdown
ax3 = axes[1, 0]
dataset_info = ['Easy\nQuestions', 'Challenge\nQuestions', 'Total\nSamples']
dataset_values = [15, 15, benchmark_results['Total Samples']]
colors3 = ['lightblue', 'lightcoral', 'lightgreen']
bars3 = ax3.bar(dataset_info, dataset_values, color=colors3, alpha=0.7, edgecolor='black', linewidth=1.5)
ax3.set_ylabel('Count', fontsize=11, fontweight="bold")
ax3.set_title('Dataset Composition', fontsize=12, fontweight="bold")
ax3.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars3, dataset_values):
    height = bar.get_height()
    ax3.text(bar.get_x() + bar.get_width()/2., height, f'{int(val)}', 
            ha='center', va='bottom', fontsize=10, fontweight='bold')

# Plot 4: Summary statistics table
ax4 = axes[1, 1]
ax4.axis("off")

summary_text = f"""
BENCHMARK SUMMARY

Dataset: ARC (AI2 Reasoning Challenge)
Model: DistilBERT (Base)
Device: {'T4 GPU' if torch.cuda.is_available() else 'CPU'}

ACCURACY RESULTS:
• Per-Question: {benchmark_results['Accuracy (per question)'] * 100:.2f}%
• Per-Choice: {benchmark_results['Accuracy (per choice)'] * 100:.2f}%

PERFORMANCE METRICS:
• Avg Inference: {benchmark_results['Avg Inference Time (ms)']:.2f} ms
• Throughput: {benchmark_results['Total Samples'] / benchmark_results['Total Inference Time (s)']:.2f} samples/s
• Total Time: {benchmark_results['Total Inference Time (s)']:.2f}s

DATASET STATS:
• Questions: {benchmark_results['Total Questions']}
• Total Samples: {benchmark_results['Total Samples']}
• Model Size: {param_size:.2f} MB

Note: The model was trained on sequence classification,
not specifically for ARC. Results show reasoning capability.
"""

ax4.text(0.1, 0.9, summary_text, transform=ax4.transAxes, fontsize=9.5,
        verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

print("\n✓ Visualization complete!")

## About This Benchmark

### What is ARC?
The **AI2 Reasoning Challenge** (ARC) is a benchmark containing science exam questions from 3rd to 9th grade. Questions span:
- **Physics**: Motion, Forces, Energy
- **Chemistry**: Elements, Reactions, Matter
- **Biology**: Life Processes, Genetics, Ecosystems
- **Earth Science**: Weather, Geology, Astronomy
- **General Science**: Problem-Solving, Analysis

### Dataset Details
- **ARC-Easy**: 2,590 questions (easier difficulty)
- **ARC-Challenge**: 1,119 questions (harder difficulty)
- **Format**: Multiple choice with 4 answer options (A, B, C, D)
- **This Benchmark**: Uses 15 Easy + 15 Challenge questions for quick testing

### How This Benchmark Works
1. Each question-answer pair is evaluated separately
2. Model predicts confidence for each choice being correct
3. Highest confidence choice is selected as the answer
4. Accuracy is calculated against ground truth labels

### Note on DistilBERT
- DistilBERT was trained on **sequence classification** tasks (like sentiment analysis)
- It's **not specifically trained** for science QA or ARC
- This benchmark shows general reasoning and knowledge retention capabilities
- For better ARC performance, consider fine-tuning or using a QA-specialized model

### How to Use in Google Colab
1. Open [Google Colab](https://colab.research.google.com)
2. Go to **Runtime → Change runtime type → GPU (T4)**
3. Upload this notebook
4. Run all cells: **Runtime → Run all**
5. View accuracy results and performance metrics

### Customization
To use more ARC questions, modify line in "Load ARC Dataset" cell:
```python
arc_easy = load_dataset("ai2_arc", "ARC-Easy", split="test", trust_remote_code=True)[:N]  # Change N
arc_challenge = load_dataset("ai2_arc", "ARC-Challenge", split="test", trust_remote_code=True)[:N]  # Change N
```

### Tips for Better Results
- **Fine-tuning**: Train the model on ARC questions for better accuracy
- **Larger models**: Try RoBERTa or ALBERT for improved reasoning
- **Ensemble**: Combine multiple models for better predictions
- **Prompt engineering**: Reformulate questions to improve understanding